# Laboratorio Hadoop + Hive

Este cuaderno recorre el flujo basico: subir datos a **HDFS**, crear una tabla externa en **Hive** y analizarla con SQL y pandas.

Servicios (red interna de Docker): `namenode:9870` (WebHDFS) y `hive-server:10000` (HiveServer2).

## 1. Subir datos a HDFS (WebHDFS)

In [ ]:
from hdfs import InsecureClient

hdfs = InsecureClient("http://namenode:9870", user="root")
hdfs.makedirs("/datasets/ventas")
hdfs.upload("/datasets/ventas/ventas.csv", "/home/jovyan/data/ventas.csv", overwrite=True)
print(hdfs.list("/datasets/ventas"))
print(hdfs.status("/datasets/ventas/ventas.csv"))

## 2. Conectar con Hive

In [ ]:
from pyhive import hive
import pandas as pd

conn = hive.Connection(host="hive-server", port=10000, username="root", auth="NONE")
cur = conn.cursor()

def q(sql):
    """Ejecuta una consulta y devuelve un DataFrame."""
    cur.execute(sql)
    cols = [c[0].split(".")[-1] for c in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

cur.execute("CREATE DATABASE IF NOT EXISTS tienda")
q("SHOW DATABASES")

## 3. Tabla externa sobre el CSV de HDFS

In [ ]:
cur.execute("DROP TABLE IF EXISTS tienda.ventas")
cur.execute("""
CREATE EXTERNAL TABLE tienda.ventas (
  id INT, fecha DATE, tienda STRING, categoria STRING,
  producto STRING, unidades INT, precio DOUBLE
)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION '/datasets/ventas'
TBLPROPERTIES ('skip.header.line.count'='1')
""")
q("SELECT * FROM tienda.ventas LIMIT 5")

## 4. Analisis con SQL

In [ ]:
ingresos = q("""
SELECT categoria, SUM(unidades * precio) AS ingresos, SUM(unidades) AS unidades
FROM tienda.ventas
GROUP BY categoria
ORDER BY ingresos DESC
""")
ingresos

In [ ]:
ingresos.plot.bar(x="categoria", y="ingresos", legend=False, title="Ingresos por categoria")

## 5. Siguientes pasos

- Abre **Hue** (http://localhost:8888) y ejecuta la misma consulta desde el editor de Hive.
- Mira la ejecucion en la UI de YARN (http://localhost:8088) lanzando el `wordcount` del README.
- Crea una tabla `STORED AS PARQUET` con `CREATE TABLE ... AS SELECT` y compara tamanos en HDFS.
- Prueba a particionar por `tienda`.